In [11]:
# ----------------------------------
# Required libraries
# ----------------------------------
import pandas as pd
import numpy as np

In [12]:
# ----------------------------------
# Read in claims dataset
# Make sure path matches your folder structure
# ----------------------------------

claims_data = pd.read_excel("../../data/raw/ClaimDetails_for_distribution.xlsx")

# Quick check
claims_data.head()

,Weather,Territory,Claim Number,CAT Code,CAT Severity Code,Loss Date,NOL Date,Peril Description,Cause Of Injury Text,Peril Group,...,6_cluster_cluster_id,6_cluster_distance_to_centroid_km,7_cluster_cluster_id,7_cluster_distance_to_centroid_km,8_cluster_cluster_id,8_cluster_distance_to_centroid_km,9_cluster_cluster_id,9_cluster_distance_to_centroid_km,10_cluster_cluster_id,10_cluster_distance_to_centroid_km
0,Weather,Alabama - Middle ...,25266097,82,3,2023-12-09,2023-12-10,Wind,TORNADO CAME THROUGH AND TREE WENT THROUGH INS...,Wind,...,1,104.359385,1,104.359385,1,1.876188,1,1.876188,1,1.876188
1,Weather,Alabama - Middle ...,86498344,82,5,2023-12-10,2023-12-10,Wind,"HEAVY STORMS WITH WIND, HAIL, AND TREES DOWN. ...",Wind,...,1,104.307219,1,104.307219,1,1.917308,1,1.917308,1,1.917308
2,Weather,Alabama - Middle ...,11851998,82,5,2023-12-10,2023-12-11,Wind,TREE FELL ON HOME FROM BAD WEATHER,Wind,...,1,97.903774,1,97.903774,1,4.737880,1,4.737880,1,4.737880
3,Weather,Alabama - North ...,15735237,82,2,2023-12-09,2023-12-21,Hail,"A HAILSTORM HIT OUR AREA-IT DAMAGED THE ROOF, ...",Hail,...,3,188.857257,3,188.857257,1,166.455853,1,166.455853,1,166.455853
4,Weather,Metro Atlanta - West ...,50814161,82,3,2023-12-10,2023-12-10,Wind,****DRP ELIGIBLE****WIND,Wind,...,1,115.875457,1,115.875457,7,53.899343,7,42.586919,7,42.586919


In [6]:
# ----------------------------------
# Random Assignment Generator
# ----------------------------------

# add a variable to the function to filter in the day
# randomize/filter for just that single day
# so randomizer should work for just the single day in the moment

# Set seed for reproducibility (same as set.seed(123))
np.random.seed(123)

def assign_adjustment(severity):
    
    # Handle missing values safely
    if pd.isna(severity):
        return np.nan
    
    # Severity 1 → Always Virtual
    if severity == 1:
        return "Virtual"
    
    # Severity 2 → 50/50 random
    if severity == 2:
        return np.random.choice(
            ["Virtual", "In-Person"],
            p=[0.5, 0.5]
        )
    
    # Severity 3 → 25% Virtual, 75% In-Person
    if severity == 3:
        return np.random.choice(
            ["Virtual", "In-Person"],
            p=[0.25, 0.75]
        )
    
    # Severity 4 or 5 → Always In-Person
    if severity in [4, 5]:
        return "In-Person"

# Apply function to dataset
claims_data["Assignment"] = claims_data["CAT Severity Code"].apply(assign_adjustment)

In [7]:
# ----------------------------------
# Row-wise proportions by severity
# ----------------------------------

pd.crosstab(
    claims_data["CAT Severity Code"],
    claims_data["Assignment"],
    normalize="index"   # row-wise normalization
)

Assignment,In-Person,Virtual
CAT Severity Code,,
1,0.000000,1.000000
2,0.511111,0.488889
3,0.752381,0.247619
4,1.000000,0.000000
5,1.000000,0.000000


In [17]:
# ----------------------------------
# Random Assignment Generator for a Single Day
# ----------------------------------
import numpy as np
import pandas as pd

def assign_adjustment_daily(df):
    """
    Assign Virtual/In-Person per NOL Date batch.
    Each day is processed independently.
    """
    df = df.copy()
    
    # Ensure datetime
    df["NOL Date"] = pd.to_datetime(df["NOL Date"])
    
    # Create assignment column if not present
    if "Assignment" not in df.columns:
        df["Assignment"] = np.nan
    
    def assign(severity):
        if pd.isna(severity):
            return np.nan
        if severity == 1:
            return "Virtual"
        if severity == 2:
            return np.random.choice(["Virtual", "In-Person"], p=[0.5, 0.5])
        if severity == 3:
            return np.random.choice(["Virtual", "In-Person"], p=[0.25, 0.75])
        if severity in [4, 5]:
            return "In-Person"
    
    # Process each day separately
    for day in df["NOL Date"].unique():
        
        # Optional: set seed based on day for reproducibility
        np.random.seed(int(day.strftime("%Y%m%d")))
        
        mask = df["NOL Date"] == day
        df.loc[mask, "Assignment"] = df.loc[mask, "CAT Severity Code"].apply(assign)
    
    return df

In [18]:
claims_data = assign_adjustment_daily(claims_data)

claims_data[["NOL Date", "CAT Severity Code", "Assignment"]].head()

,NOL Date,CAT Severity Code,Assignment
0,2023-12-10,3,In-Person
1,2023-12-10,5,In-Person
2,2023-12-11,5,In-Person
3,2023-12-21,2,Virtual
4,2023-12-10,3,Virtual
